# Temporal Models for HbA1c Diabetes Fraud Detection

This notebook implements two temporal models for diabetes claims:

1. **Model 3: Patient Temporal LSTM Autoencoder** - Detects anomalous diabetic patient trajectories over time
   - Captures patterns like sudden HbA1c spikes or drops
   - Identifies deviations from patient's own baseline lab values (HbA1c, Creatinine, Urea)
   - Flags unusual treatment patterns for diabetic patients

2. **Model 4: Facility Temporal LSTM Autoencoder** - Detects anomalous facility behavior over time
   - Identifies sudden spikes in diabetic admissions
   - Detects unusual patterns in HbA1c distributions
   - Flags temporal clustering of diabetic complications

## Dataset Requirements
- Patient sequences: Multiple diabetic claims per patient over time
- Facility aggregates: Weekly statistics per facility for diabetic claims

## 1. Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import json
from datetime import datetime, timedelta
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Set random seeds
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# 1. Anchor to the project root (bypassing the app/resultshield_lite nesting)
BASE_DIR = Path.home() / "localfiles" / "lstm"

# 2. Main Directory Paths
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
PLOTS_DIR = BASE_DIR / "visualizations" / "plots"

# 3. Specific File Paths for Diabetes Dataset
DATA_PATH = DATA_DIR / "kenyan_diabetic_claims_20260321_145103.csv"

# 4. Ensure directories exist
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# 5. Global Toggle - Set to False for full 3M training
TEST_MODE = False  # False = Full training on 3M rows

print("=" * 60)
print("  Temporal Models for HbA1c Diabetes Fraud Detection")
print("=" * 60)
print(f"  TensorFlow : {tf.__version__}")
print(f"  GPU available: {tf.config.list_physical_devices('GPU')}")
print(f"  Data       : {DATA_PATH}")
print(f"  Models dir : {MODEL_DIR}")
print(f"  Plots dir  : {PLOTS_DIR}")
print(f"  Test Mode  : {TEST_MODE}")
if TEST_MODE:
    print(f"  Mode       : TESTING (using 500 samples)")
else:
    print(f"  Mode       : FULL TRAINING (3M rows)")
print("=" * 60)

2026-03-21 17:55:45.074487: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 17:55:45.170589: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-21 17:55:45.170631: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-21 17:55:45.172303: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-21 17:55:45.185773: I tensorflow/core/platform/cpu_feature_guar

  Temporal Models for HbA1c Diabetes Fraud Detection
  TensorFlow : 2.15.0
  GPU available: []
  Data       : /home/azureuser/localfiles/lstm/data/kenyan_diabetic_claims_20260321_164001.csv
  Models dir : /home/azureuser/localfiles/lstm/models
  Plots dir  : /home/azureuser/localfiles/lstm/visualizations/plots
  Test Mode  : False
  Mode       : FULL TRAINING (3M rows)


2026-03-21 17:55:49.714472: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:274] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


## 2. Load and Prepare Diabetes Data

In [2]:
# Load dataset
print("\n📂 Loading diabetes dataset...")
df_full = pd.read_csv(DATA_PATH)
print(f"Full dataset shape: {df_full.shape}")

# Convert date columns
df_full['date'] = pd.to_datetime(df_full['date'])
df_full['admission_date'] = pd.to_datetime(df_full['admission_date'])
df_full['discharge_date'] = pd.to_datetime(df_full['discharge_date'])

# Sort by date for patient sequences
df_full = df_full.sort_values(['patient_id', 'date']).reset_index(drop=True)

print(f"\nDataset Info:")
print(f"  Claims: {len(df_full):,}")
print(f"  Patients: {df_full['patient_id'].nunique():,}")
print(f"  Facilities: {df_full['facility_id'].nunique():,}")
print(f"  Date range: {df_full['date'].min()} to {df_full['date'].max()}")

# Sample for testing
if TEST_MODE:
    # Get patients with multiple visits for testing
    patient_visit_counts = df_full.groupby('patient_id').size()
    patients_with_multiple = patient_visit_counts[patient_visit_counts >= 3].index.tolist()
    
    if len(patients_with_multiple) > 0:
        sample_patients = np.random.choice(patients_with_multiple, size=min(50, len(patients_with_multiple)), replace=False)
        df = df_full[df_full['patient_id'].isin(sample_patients)].copy()
    else:
        df = df_full.sample(n=500, random_state=RANDOM_SEED).copy()
    
    print(f"\n🧪 TEST MODE: Using {len(df)} claims from {df['patient_id'].nunique()} patients")
else:
    df = df_full.copy()
    print(f"\n🚀 FULL TRAINING MODE: Using {len(df):,} claims")

# Feature engineering
df['sex_encoded'] = (df['sex'] == 'Male').astype(int)
df['length_of_stay'] = (df['discharge_date'] - df['admission_date']).dt.days
df['length_of_stay'] = df['length_of_stay'].clip(0, 30)  # Cap at 30 days

# HbA1c specific lab features
LAB_FEATURES = ['HBA1C', 'CREATININE', 'UREA']
DEMO_FEATURES = ['age', 'sex_encoded']
TEMPORAL_FEATURES = ['length_of_stay']

# Complete feature set for patient model
PATIENT_FEATURES = DEMO_FEATURES + LAB_FEATURES + TEMPORAL_FEATURES
N_PATIENT_FEATURES = len(PATIENT_FEATURES)

print(f"\n📊 Patient-level features ({N_PATIENT_FEATURES}):")
print(f"  Demographics: {DEMO_FEATURES}")
print(f"  Lab values: {LAB_FEATURES}")
print(f"  Temporal: {TEMPORAL_FEATURES}")

# Handle missing values
print("\n📊 Missing values before handling:")
print(df[PATIENT_FEATURES].isnull().sum())
df[PATIENT_FEATURES] = df[PATIENT_FEATURES].fillna(df[PATIENT_FEATURES].median())


📂 Loading diabetes dataset...


FileNotFoundError: [Errno 2] No such file or directory: '/home/azureuser/localfiles/lstm/data/kenyan_diabetic_claims_20260321_164001.csv'

## 3. Model 3: Patient Temporal LSTM Autoencoder (Diabetes)

### Objective
- Learn normal diabetic patient trajectories over time
- Detect anomalous visits (e.g., sudden HbA1c changes, kidney function deterioration)
- Input: Sequences of patient visits (fixed length)
- Output: Reconstructed sequences
- Anomaly score = sequence reconstruction error

In [ ]:
# ==================== CREATE PATIENT SEQUENCES (OPTIMIZED FOR 3M ROWS) ====================

def create_patient_sequences_fast(df, patient_features, seq_len=5):
    """
    Optimized creation of patient visit sequences using numpy arrays.
    ~100x faster than loop version for large datasets.
    """
    X_sequences = []
    patient_ids = []
    sequence_dates = []
    
    # Group by patient
    for patient_id, group in df.groupby('patient_id'):
        if len(group) < seq_len:
            continue
        
        # Sort by date
        group = group.sort_values('date')
        
        # Extract features
        features = group[patient_features].values
        
        # Create sliding windows
        for i in range(len(group) - seq_len + 1):
            seq = features[i:i+seq_len]
            X_sequences.append(seq)
            patient_ids.append(patient_id)
            sequence_dates.append(group.iloc[i+seq_len-1]['date'])
    
    return np.array(X_sequences), patient_ids, sequence_dates

# Sequence parameters
PATIENT_SEQ_LEN = 5  # Number of visits per sequence

print(f"\nCreating patient sequences (length={PATIENT_SEQ_LEN})...")

# Scale features
scaler_patient = StandardScaler()
df_patient_scaled = df.copy()
df_patient_scaled[PATIENT_FEATURES] = scaler_patient.fit_transform(df[PATIENT_FEATURES])

# Create sequences
X_patient_seq, patient_ids, seq_dates = create_patient_sequences_fast(
    df_patient_scaled, PATIENT_FEATURES, PATIENT_SEQ_LEN
)

print(f"✅ Created {len(X_patient_seq):,} sequences from {len(np.unique(patient_ids)):,} patients")
print(f"  Sequence shape: {X_patient_seq.shape}")
print(f"  Features per visit: {N_PATIENT_FEATURES}")

# Save scaler
scaler_patient_path = MODEL_DIR / "diabetes_model3_patient_scaler.pkl"
joblib.dump(scaler_patient, scaler_patient_path)
print(f"✅ Patient scaler saved to {scaler_patient_path}")

In [ ]:
# ==================== OPTIMIZED TRAIN/VAL/TEST SPLIT ====================

# Convert to sets for O(1) lookup
unique_patients = np.unique(patient_ids)
train_patients, temp_patients = train_test_split(
    unique_patients, test_size=0.3, random_state=RANDOM_SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=0.5, random_state=RANDOM_SEED
)

# Convert to sets for fast membership testing
train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print("Creating train indices...")
train_idx = [i for i, pid in enumerate(patient_ids) if pid in train_set]

print("Creating validation indices...")
val_idx = [i for i, pid in enumerate(patient_ids) if pid in val_set]

print("Creating test indices...")
test_idx = [i for i, pid in enumerate(patient_ids) if pid in test_set]

X_patient_train = X_patient_seq[train_idx]
X_patient_val = X_patient_seq[val_idx]
X_patient_test = X_patient_seq[test_idx]

print(f"\n✅ Split complete!")
print(f"Train sequences: {X_patient_train.shape[0]:,} (from {len(train_patients):,} patients)")
print(f"Val sequences:   {X_patient_val.shape[0]:,} (from {len(val_patients):,} patients)")
print(f"Test sequences:  {X_patient_test.shape[0]:,} (from {len(test_patients):,} patients)")
print(f"\nTotal sequences: {len(X_patient_seq):,}")

In [ ]:
# ==================== BUILD PATIENT LSTM AUTOENCODER (SCALED FOR 3M ROWS) ====================

def build_patient_lstm_autoencoder(seq_len, n_features, latent_dim=48):
    """
    Build LSTM autoencoder for diabetic patient temporal analysis.
    
    Optimized for 3M rows with increased capacity:
    - Larger LSTM units for complex diabetic pattern learning
    - Deeper architecture for hierarchical feature extraction
    - Balanced dropout for regularization
    """
    
    # Encoder
    encoder_input = layers.Input(shape=(seq_len, n_features), name='encoder_input')
    
    # First LSTM layer - wider for better feature extraction
    x = layers.LSTM(128, activation='tanh', return_sequences=True, name='encoder_lstm_1')(encoder_input)
    x = layers.Dropout(0.3, name='encoder_dropout_1')(x)
    x = layers.BatchNormalization(name='encoder_bn_1')(x)
    
    # Second LSTM layer - medium width
    x = layers.LSTM(64, activation='tanh', return_sequences=True, name='encoder_lstm_2')(x)
    x = layers.Dropout(0.3, name='encoder_dropout_2')(x)
    x = layers.BatchNormalization(name='encoder_bn_2')(x)
    
    # Third LSTM layer - final hidden state
    x = layers.LSTM(32, activation='tanh', return_sequences=False, name='encoder_lstm_3')(x)
    x = layers.Dropout(0.2, name='encoder_dropout_3')(x)
    
    # Bottleneck - increased latent dimension for complex patterns
    latent = layers.Dense(latent_dim, activation='relu', name='bottleneck')(x)
    
    # Decoder
    x = layers.RepeatVector(seq_len, name='repeat')(latent)
    
    # First decoder LSTM
    x = layers.LSTM(32, activation='tanh', return_sequences=True, name='decoder_lstm_1')(x)
    x = layers.Dropout(0.2, name='decoder_dropout_1')(x)
    x = layers.BatchNormalization(name='decoder_bn_1')(x)
    
    # Second decoder LSTM
    x = layers.LSTM(64, activation='tanh', return_sequences=True, name='decoder_lstm_2')(x)
    x = layers.Dropout(0.3, name='decoder_dropout_2')(x)
    x = layers.BatchNormalization(name='decoder_bn_2')(x)
    
    # Third decoder LSTM
    x = layers.LSTM(128, activation='tanh', return_sequences=True, name='decoder_lstm_3')(x)
    x = layers.Dropout(0.3, name='decoder_dropout_3')(x)
    
    # Output layer
    decoder_output = layers.TimeDistributed(
        layers.Dense(n_features, activation='linear'), 
        name='decoder_output'
    )(x)
    
    # Model
    autoencoder = models.Model(encoder_input, decoder_output, name='diabetes_patient_temporal_ae')
    
    return autoencoder

# Model parameters
PATIENT_LATENT_DIM = 48
PATIENT_LEARNING_RATE = 0.001

# Build model
model3 = build_patient_lstm_autoencoder(
    PATIENT_SEQ_LEN, 
    N_PATIENT_FEATURES, 
    PATIENT_LATENT_DIM
)

# Compile
model3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=PATIENT_LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

print(f"\n📊 Model3 Configuration for 3M rows:")
print(f"  Sequence length: {PATIENT_SEQ_LEN}")
print(f"  Features per timestep: {N_PATIENT_FEATURES}")
print(f"  Latent dimension: {PATIENT_LATENT_DIM}")
print(f"  Total trainable parameters: {model3.count_params():,}")
model3.summary()

In [ ]:
# ==================== TRAIN PATIENT MODEL (OPTIMIZED FOR 3M ROWS) ====================

# Training parameters
EPOCHS_PATIENT = 30 if TEST_MODE else 80      # More epochs for convergence
BATCH_SIZE_PATIENT = 32 if TEST_MODE else 512  # Larger batches for 3M rows
PATIENCE_PATIENT = 10                          # More patience for large dataset

# Callbacks
model3_path = MODEL_DIR / "diabetes_model3_patient_temporal_ae.keras"

callbacks3 = [
    tf.keras.callbacks.ModelCheckpoint(
        model3_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE_PATIENT,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        MODEL_DIR / "diabetes_model3_training_log.csv"
    )
]

print("\n🚀 Starting Model 3 (Diabetes Patient Temporal) training...\n")
print(f"Training Configuration:")
print(f"  - Epochs: {EPOCHS_PATIENT}")
print(f"  - Batch size: {BATCH_SIZE_PATIENT}")
print(f"  - Training sequences: {len(X_patient_train):,}")
print(f"  - Validation sequences: {len(X_patient_val):,}")
print(f"  - Early stopping patience: {PATIENCE_PATIENT}")

history3 = model3.fit(
    X_patient_train, X_patient_train,
    validation_data=(X_patient_val, X_patient_val),
    epochs=EPOCHS_PATIENT,
    batch_size=BATCH_SIZE_PATIENT,
    callbacks=callbacks3,
    verbose=1,
    shuffle=True
)

print(f"\n✅ Model 3 training complete!")
best_val_loss = min(history3.history['val_loss'])
print(f"Best val_loss: {best_val_loss:.6f}")

In [ ]:
# ==================== PLOT PATIENT MODEL TRAINING ====================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax = axes[0]
ax.plot(history3.history['loss'], label='Training Loss', linewidth=2)
ax.plot(history3.history['val_loss'], label='Validation Loss', linewidth=2)
best_epoch = np.argmin(history3.history['val_loss'])
ax.scatter(best_epoch, history3.history['val_loss'][best_epoch], 
          color='red', s=100, zorder=5, label=f'Best: {history3.history["val_loss"][best_epoch]:.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Model 3: Diabetes Patient Temporal LSTM Autoencoder')
ax.legend()
ax.grid(True, alpha=0.3)

# MAE
ax = axes[1]
ax.plot(history3.history['mae'], label='Training MAE', linewidth=2)
ax.plot(history3.history['val_mae'], label='Validation MAE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.set_title('Training and Validation MAE')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Save
plot_path = PLOTS_DIR / "diabetes_model3_patient_training.png"
plt.savefig(plot_path, dpi=100, bbox_inches='tight')
plt.close()
print(f"✅ Plot saved to {plot_path}")

In [ ]:
# ==================== DETERMINE PATIENT ANOMALY THRESHOLD ====================

# Predict on validation set
X_patient_val_pred = model3.predict(X_patient_val, verbose=0)

# Calculate reconstruction errors (MSE per sequence)
recon_errors_patient_val = np.mean(np.square(X_patient_val - X_patient_val_pred), axis=(1, 2))

# Set threshold at 95th percentile
PATIENT_THRESHOLD_PERCENTILE = 95
patient_threshold = np.percentile(recon_errors_patient_val, PATIENT_THRESHOLD_PERCENTILE)

print(f"\n📊 Patient Sequence Reconstruction Errors (Validation):")
print(f"  Mean: {np.mean(recon_errors_patient_val):.6f}")
print(f"  Std:  {np.std(recon_errors_patient_val):.6f}")
print(f"  Min:  {np.min(recon_errors_patient_val):.6f}")
print(f"  Max:  {np.max(recon_errors_patient_val):.6f}")
print(f"\n🎯 Patient Anomaly Threshold ({PATIENT_THRESHOLD_PERCENTILE}th percentile): {patient_threshold:.6f}")

In [ ]:
# ==================== EVALUATE PATIENT MODEL ON TEST SET ====================

# Predict on test set
X_patient_test_pred = model3.predict(X_patient_test, verbose=0)

# Calculate reconstruction errors
recon_errors_patient_test = np.mean(np.square(X_patient_test - X_patient_test_pred), axis=(1, 2))

# Detect anomalies
patient_anomalies = (recon_errors_patient_test > patient_threshold).astype(int)

print(f"\n📊 Test Set Results:")
print(f"  Total sequences: {len(X_patient_test):,}")
print(f"  Detected anomalies: {np.sum(patient_anomalies)}")
print(f"  Anomaly rate: {np.mean(patient_anomalies):.2%}")

In [ ]:
# ==================== SAVE MODEL 3 METADATA ====================

metadata3 = {
    'model_name': 'Model 3: Diabetes Patient Temporal LSTM Autoencoder',
    'model_type': 'lstm_autoencoder',
    'test_type': 'HbA1c',
    'sequence_length': PATIENT_SEQ_LEN,
    'n_features': N_PATIENT_FEATURES,
    'feature_names': PATIENT_FEATURES,
    'latent_dim': PATIENT_LATENT_DIM,
    'threshold': float(patient_threshold),
    'threshold_percentile': PATIENT_THRESHOLD_PERCENTILE,
    'train_sequences': len(X_patient_train),
    'val_sequences': len(X_patient_val),
    'test_sequences': len(X_patient_test),
    'train_patients': len(train_patients),
    'val_patients': len(val_patients),
    'test_patients': len(test_patients),
    'best_val_loss': float(min(history3.history['val_loss'])),
    'test_anomaly_rate': float(np.mean(patient_anomalies)),
    'training_date': datetime.now().isoformat(),
    'epochs_trained': len(history3.history['loss']),
    'batch_size': BATCH_SIZE_PATIENT,
    'learning_rate': PATIENT_LEARNING_RATE
}

# Save metadata
meta3_path = MODEL_DIR / "diabetes_model3_patient_temporal_meta.json"
with open(meta3_path, 'w') as f:
    json.dump(metadata3, f, indent=2)

print(f"✅ Model 3 saved to: {model3_path}")
print(f"✅ Metadata saved to: {meta3_path}")

## 4. Model 4: Facility Temporal LSTM Autoencoder (Diabetes)

### Objective
- Learn normal diabetic facility behavior patterns over time
- Detect anomalous weeks (e.g., sudden spike in diabetic admissions, abnormal HbA1c distributions)
- Input: Weekly aggregated statistics per facility for diabetes claims
- Output: Reconstructed weekly patterns
- Anomaly score = week reconstruction error

In [ ]:
# ==================== CREATE FACILITY WEEKLY AGGREGATES (DIABETES) ====================

def create_facility_weekly_aggregates(df):
    """
    Create weekly aggregates per facility for diabetes claims.
    """
    # Create week column
    df['week'] = df['date'].dt.isocalendar().week.astype(str) + '-' + df['date'].dt.year.astype(str)
    df['week_num'] = ((df['date'] - df['date'].min()).dt.days // 7).astype(int)
    
    # Group by facility and week
    facility_weekly = df.groupby(['facility_id', 'facility_name', 'week_num']).agg({
        'claim_id': 'count',
        'age': ['mean', 'std'],
        'sex_encoded': 'mean',
        'HBA1C': ['mean', 'std'],
        'CREATININE': ['mean', 'std'],
        'UREA': ['mean', 'std'],
        'length_of_stay': 'mean'
    }).reset_index()
    
    # Flatten column names
    facility_weekly.columns = ['_'.join(col).strip() if col[1] else col[0] 
                               for col in facility_weekly.columns.values]
    
    # Rename for clarity
    facility_weekly = facility_weekly.rename(columns={
        'claim_id_count': 'claim_volume',
        'age_mean': 'avg_age',
        'age_std': 'age_std',
        'sex_encoded_mean': 'pct_male',
        'length_of_stay_mean': 'avg_los'
    })
    
    return facility_weekly

# Create facility weekly aggregates
print("\nCreating facility weekly aggregates for diabetes claims...")
facility_weekly = create_facility_weekly_aggregates(df)

print(f"✅ Created {len(facility_weekly)} facility-week records")
print(f"  Facilities: {facility_weekly['facility_id'].nunique()}")
print(f"  Weeks: {facility_weekly['week_num'].min()} to {facility_weekly['week_num'].max()}")

In [ ]:
# ==================== CREATE FACILITY TIME SERIES ====================

def create_facility_time_series(facility_weekly, seq_len=8):
    """
    Create time series sequences for each facility.
    """
    # Select feature columns (exclude identifiers)
    feature_cols = [col for col in facility_weekly.columns 
                    if col not in ['facility_id', 'facility_name', 'week_num']]
    
    n_features = len(feature_cols)
    
    # Pivot to get time series per facility
    facilities = facility_weekly['facility_id'].unique()
    weeks = sorted(facility_weekly['week_num'].unique())
    
    # Create 3D array: (facility, week, feature)
    n_facilities = len(facilities)
    n_weeks = len(weeks)
    
    data_3d = np.full((n_facilities, n_weeks, n_features), np.nan)
    
    # Fill array
    for i, facility in enumerate(facilities):
        fac_data = facility_weekly[facility_weekly['facility_id'] == facility]
        for j, week in enumerate(weeks):
            row = fac_data[fac_data['week_num'] == week]
            if not row.empty:
                data_3d[i, j, :] = row[feature_cols].values[0]
    
    # Handle missing weeks (linear interpolation)
    from scipy.interpolate import interp1d
    
    for i in range(n_facilities):
        for f in range(n_features):
            series = data_3d[i, :, f]
            mask = ~np.isnan(series)
            
            if np.sum(mask) > 1:
                x = np.arange(n_weeks)[mask]
                y = series[mask]
                f_interp = interp1d(x, y, kind='linear', fill_value='extrapolate')
                data_3d[i, :, f] = f_interp(np.arange(n_weeks))
            elif np.sum(mask) == 1:
                data_3d[i, :, f] = series[mask][0]
            else:
                data_3d[i, :, f] = 0
    
    # Create sequences (sliding window per facility)
    X_sequences = []
    sequence_facilities = []
    sequence_weeks = []
    
    for i, facility in enumerate(facilities):
        for j in range(n_weeks - seq_len + 1):
            seq = data_3d[i, j:j+seq_len, :]
            X_sequences.append(seq)
            sequence_facilities.append(facility)
            sequence_weeks.append(weeks[j+seq_len-1])
    
    return np.array(X_sequences), sequence_facilities, sequence_weeks, feature_cols

# Sequence parameters
FACILITY_SEQ_LEN = 8  # Number of weeks per sequence

print(f"\nCreating facility time series (sequence length={FACILITY_SEQ_LEN} weeks)...")

# Scale features
feature_cols = [col for col in facility_weekly.columns 
                if col not in ['facility_id', 'facility_name', 'week_num']]
scaler_facility = StandardScaler()
facility_weekly_scaled = facility_weekly.copy()
facility_weekly_scaled[feature_cols] = scaler_facility.fit_transform(facility_weekly[feature_cols])

# Create sequences
X_facility_seq, facility_ids, facility_weeks, facility_feature_names = create_facility_time_series(
    facility_weekly_scaled, FACILITY_SEQ_LEN
)

N_FACILITY_FEATURES = len(facility_feature_names)

print(f"✅ Created {len(X_facility_seq):,} facility sequences")
print(f"  Sequence shape: {X_facility_seq.shape}")
print(f"  Features per week: {N_FACILITY_FEATURES}")
print(f"  Facilities with sequences: {len(np.unique(facility_ids))}")

# Save scaler
scaler_facility_path = MODEL_DIR / "diabetes_model4_facility_scaler.pkl"
joblib.dump(scaler_facility, scaler_facility_path)
print(f"✅ Facility scaler saved to {scaler_facility_path}")

In [ ]:
# ==================== TRAIN/VAL/TEST SPLIT (BY FACILITY) ====================

# Split facilities
unique_facilities = np.unique(facility_ids)
train_facilities, temp_facilities = train_test_split(
    unique_facilities, test_size=0.3, random_state=RANDOM_SEED
)
val_facilities, test_facilities = train_test_split(
    temp_facilities, test_size=0.5, random_state=RANDOM_SEED
)

# Convert to sets for fast lookup
train_fac_set = set(train_facilities)
val_fac_set = set(val_facilities)
test_fac_set = set(test_facilities)

# Get indices for each split
train_idx = [i for i, fid in enumerate(facility_ids) if fid in train_fac_set]
val_idx = [i for i, fid in enumerate(facility_ids) if fid in val_fac_set]
test_idx = [i for i, fid in enumerate(facility_ids) if fid in test_fac_set]

X_facility_train = X_facility_seq[train_idx]
X_facility_val = X_facility_seq[val_idx]
X_facility_test = X_facility_seq[test_idx]

print(f"\nTrain sequences: {X_facility_train.shape} (from {len(train_facilities):,} facilities)")
print(f"Val sequences:   {X_facility_val.shape} (from {len(val_facilities):,} facilities)")
print(f"Test sequences:  {X_facility_test.shape} (from {len(test_facilities):,} facilities)")

In [ ]:
# ==================== BUILD FACILITY LSTM AUTOENCODER (SCALED FOR 3M ROWS) ====================

def build_facility_lstm_autoencoder(seq_len, n_features, latent_dim=32):
    """
    Build LSTM autoencoder for facility temporal analysis.
    Optimized for 3M rows with increased capacity.
    """
    
    # Encoder
    encoder_input = layers.Input(shape=(seq_len, n_features), name='encoder_input')
    
    # First LSTM layer
    x = layers.LSTM(128, activation='tanh', return_sequences=True, name='encoder_lstm_1')(encoder_input)
    x = layers.Dropout(0.3, name='encoder_dropout_1')(x)
    x = layers.BatchNormalization(name='encoder_bn_1')(x)
    
    # Second LSTM layer
    x = layers.LSTM(64, activation='tanh', return_sequences=True, name='encoder_lstm_2')(x)
    x = layers.Dropout(0.3, name='encoder_dropout_2')(x)
    x = layers.BatchNormalization(name='encoder_bn_2')(x)
    
    # Third LSTM layer - final hidden state
    x = layers.LSTM(32, activation='tanh', return_sequences=False, name='encoder_lstm_3')(x)
    x = layers.Dropout(0.2, name='encoder_dropout_3')(x)
    
    # Bottleneck
    latent = layers.Dense(latent_dim, activation='relu', name='bottleneck')(x)
    
    # Decoder
    x = layers.RepeatVector(seq_len, name='repeat')(latent)
    
    # First decoder LSTM
    x = layers.LSTM(32, activation='tanh', return_sequences=True, name='decoder_lstm_1')(x)
    x = layers.Dropout(0.2, name='decoder_dropout_1')(x)
    x = layers.BatchNormalization(name='decoder_bn_1')(x)
    
    # Second decoder LSTM
    x = layers.LSTM(64, activation='tanh', return_sequences=True, name='decoder_lstm_2')(x)
    x = layers.Dropout(0.3, name='decoder_dropout_2')(x)
    x = layers.BatchNormalization(name='decoder_bn_2')(x)
    
    # Third decoder LSTM
    x = layers.LSTM(128, activation='tanh', return_sequences=True, name='decoder_lstm_3')(x)
    x = layers.Dropout(0.3, name='decoder_dropout_3')(x)
    
    # Output layer
    decoder_output = layers.TimeDistributed(
        layers.Dense(n_features, activation='linear'), 
        name='decoder_output'
    )(x)
    
    # Model
    autoencoder = models.Model(encoder_input, decoder_output, name='diabetes_facility_temporal_ae')
    
    return autoencoder

# Model parameters
FACILITY_LATENT_DIM = 32
FACILITY_LEARNING_RATE = 0.001

# Build model
model4 = build_facility_lstm_autoencoder(
    FACILITY_SEQ_LEN, 
    N_FACILITY_FEATURES, 
    FACILITY_LATENT_DIM
)

# Compile
model4.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FACILITY_LEARNING_RATE),
    loss='mse',
    metrics=['mae']
)

print(f"\n📊 Model4 Configuration for 3M rows:")
print(f"  Sequence length (weeks): {FACILITY_SEQ_LEN}")
print(f"  Features per week: {N_FACILITY_FEATURES}")
print(f"  Latent dimension: {FACILITY_LATENT_DIM}")
print(f"  Total trainable parameters: {model4.count_params():,}")
model4.summary()

In [ ]:
# ==================== TRAIN FACILITY MODEL (OPTIMIZED FOR 3M ROWS) ====================

# Training parameters
EPOCHS_FACILITY = 30 if TEST_MODE else 80
BATCH_SIZE_FACILITY = 16 if TEST_MODE else 128
PATIENCE_FACILITY = 10

# Callbacks
model4_path = MODEL_DIR / "diabetes_model4_facility_temporal_ae.keras"

callbacks4 = [
    tf.keras.callbacks.ModelCheckpoint(
        model4_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE_FACILITY,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        MODEL_DIR / "diabetes_model4_training_log.csv"
    )
]

print("\n🚀 Starting Model 4 (Diabetes Facility Temporal) training...\n")
print(f"Training Configuration:")
print(f"  - Epochs: {EPOCHS_FACILITY}")
print(f"  - Batch size: {BATCH_SIZE_FACILITY}")
print(f"  - Training sequences: {len(X_facility_train):,}")
print(f"  - Validation sequences: {len(X_facility_val):,}")

history4 = model4.fit(
    X_facility_train, X_facility_train,
    validation_data=(X_facility_val, X_facility_val),
    epochs=EPOCHS_FACILITY,
    batch_size=BATCH_SIZE_FACILITY,
    callbacks=callbacks4,
    verbose=1,
    shuffle=True
)

print(f"\n✅ Model 4 training complete!")
best_val_loss = min(history4.history['val_loss'])
print(f"Best val_loss: {best_val_loss:.6f}")

In [ ]:
# ==================== DETERMINE FACILITY ANOMALY THRESHOLD ====================

# Predict on validation set
X_facility_val_pred = model4.predict(X_facility_val, verbose=0)

# Calculate reconstruction errors (MSE per sequence)
recon_errors_facility_val = np.mean(np.square(X_facility_val - X_facility_val_pred), axis=(1, 2))

# Set threshold at 95th percentile
FACILITY_THRESHOLD_PERCENTILE = 95
facility_threshold = np.percentile(recon_errors_facility_val, FACILITY_THRESHOLD_PERCENTILE)

print(f"\n📊 Facility Sequence Reconstruction Errors (Validation):")
print(f"  Mean: {np.mean(recon_errors_facility_val):.6f}")
print(f"  Std:  {np.std(recon_errors_facility_val):.6f}")
print(f"  Min:  {np.min(recon_errors_facility_val):.6f}")
print(f"  Max:  {np.max(recon_errors_facility_val):.6f}")
print(f"\n🎯 Facility Anomaly Threshold ({FACILITY_THRESHOLD_PERCENTILE}th percentile): {facility_threshold:.6f}")

In [ ]:
# ==================== EVALUATE FACILITY MODEL ON TEST SET ====================

# Predict on test set
X_facility_test_pred = model4.predict(X_facility_test, verbose=0)

# Calculate reconstruction errors
recon_errors_facility_test = np.mean(np.square(X_facility_test - X_facility_test_pred), axis=(1, 2))

# Detect anomalies
facility_anomalies = (recon_errors_facility_test > facility_threshold).astype(int)

print(f"\n📊 Test Set Results:")
print(f"  Total sequences: {len(X_facility_test):,}")
print(f"  Detected anomalies: {np.sum(facility_anomalies)}")
print(f"  Anomaly rate: {np.mean(facility_anomalies):.2%}")

In [ ]:
# ==================== SAVE MODEL 4 METADATA ====================

metadata4 = {
    'model_name': 'Model 4: Diabetes Facility Temporal LSTM Autoencoder',
    'model_type': 'lstm_autoencoder',
    'test_type': 'HbA1c',
    'sequence_length': FACILITY_SEQ_LEN,
    'n_features': N_FACILITY_FEATURES,
    'feature_names': facility_feature_names,
    'latent_dim': FACILITY_LATENT_DIM,
    'threshold': float(facility_threshold),
    'threshold_percentile': FACILITY_THRESHOLD_PERCENTILE,
    'train_sequences': len(X_facility_train),
    'val_sequences': len(X_facility_val),
    'test_sequences': len(X_facility_test),
    'train_facilities': len(train_facilities),
    'val_facilities': len(val_facilities),
    'test_facilities': len(test_facilities),
    'best_val_loss': float(min(history4.history['val_loss'])),
    'test_anomaly_rate': float(np.mean(facility_anomalies)),
    'training_date': datetime.now().isoformat(),
    'epochs_trained': len(history4.history['loss']),
    'batch_size': BATCH_SIZE_FACILITY,
    'learning_rate': FACILITY_LEARNING_RATE
}

# Save metadata
meta4_path = MODEL_DIR / "diabetes_model4_facility_temporal_meta.json"
with open(meta4_path, 'w') as f:
    json.dump(metadata4, f, indent=2)

print(f"✅ Model 4 saved to: {model4_path}")
print(f"✅ Metadata saved to: {meta4_path}")

In [ ]:
# ==================== TEST MODEL LOADING ====================

print("\n🔍 Testing Model 3 (Patient Temporal) loading and inference...")

# Load model
loaded_model3 = tf.keras.models.load_model(model3_path)
print("✅ Model 3 loaded successfully")

# Load metadata
with open(meta3_path, 'r') as f:
    loaded_meta3 = json.load(f)
print("✅ Metadata loaded successfully")

# Load scaler
loaded_scaler3 = joblib.load(scaler_patient_path)
print("✅ Scaler loaded successfully")

# Test prediction
if len(X_patient_test) > 0:
    test_samples = X_patient_test[:min(5, len(X_patient_test))]
    predictions = loaded_model3.predict(test_samples, verbose=0)
    print(f"\n📊 Test inference:")
    print(f"  Input shape: {test_samples.shape}")
    print(f"  Output shape: {predictions.shape}")

print("\n🔍 Testing Model 4 (Facility Temporal) loading and inference...")

# Load model
loaded_model4 = tf.keras.models.load_model(model4_path)
print("✅ Model 4 loaded successfully")

# Load metadata
with open(meta4_path, 'r') as f:
    loaded_meta4 = json.load(f)
print("✅ Metadata loaded successfully")

# Load scaler
loaded_scaler4 = joblib.load(scaler_facility_path)
print("✅ Scaler loaded successfully")

# Test prediction
if len(X_facility_test) > 0:
    test_samples = X_facility_test[:min(5, len(X_facility_test))]
    predictions = loaded_model4.predict(test_samples, verbose=0)
    print(f"\n📊 Test inference:")
    print(f"  Input shape: {test_samples.shape}")
    print(f"  Output shape: {predictions.shape}")

In [ ]:
# ==================== SUMMARY ====================

print("\n" + "="*60)
print("✅✅✅ DIABETES TEMPORAL MODELS TRAINED AND TESTED SUCCESSFULLY ✅✅✅")
print("="*60)
print(f"""
Model 3: Diabetes Patient Temporal LSTM Autoencoder
  - Input shape: (None, {PATIENT_SEQ_LEN}, {N_PATIENT_FEATURES})
  - Output shape: (None, {PATIENT_SEQ_LEN}, {N_PATIENT_FEATURES})
  - Latent dimension: {PATIENT_LATENT_DIM}
  - Sequences: {len(X_patient_seq):,} from {len(np.unique(patient_ids)):,} patients
  - Anomaly threshold: {patient_threshold:.6f}
  - Best validation loss: {min(history3.history['val_loss']):.6f}

Model 4: Diabetes Facility Temporal LSTM Autoencoder
  - Input shape: (None, {FACILITY_SEQ_LEN}, {N_FACILITY_FEATURES})
  - Output shape: (None, {FACILITY_SEQ_LEN}, {N_FACILITY_FEATURES})
  - Latent dimension: {FACILITY_LATENT_DIM}
  - Sequences: {len(X_facility_seq):,} from {len(np.unique(facility_ids)):,} facilities
  - Features per week: {N_FACILITY_FEATURES}
  - Anomaly threshold: {facility_threshold:.6f}
  - Best validation loss: {min(history4.history['val_loss']):.6f}

Files saved in {MODEL_DIR}:
  - diabetes_model3_patient_temporal_ae.keras
  - diabetes_model3_patient_temporal_meta.json
  - diabetes_model3_patient_scaler.pkl
  - diabetes_model4_facility_temporal_ae.keras
  - diabetes_model4_facility_temporal_meta.json
  - diabetes_model4_facility_scaler.pkl
""")